# 第7章 マルチエージェント - Supervisor パターン

役割の違う複数のエージェントを **協調させて** 1 つのタスクを片付ける構成を体験します。

**所要時間**: 60〜90分

## 学ぶこと
1. なぜ「単独エージェント」ではなく「複数エージェント」にするのか
2. **Supervisor パターン**(中央集権型)で 2 体を連携させる
3. `langgraph-supervisor` で最小コード実装
4. LangSmith で各エージェントの動きを観察

## なぜマルチエージェントなのか

第3章/第5章の「1体のエージェントがツールを次々呼ぶ」構成でも、多くのタスクは解決できます。ではなぜ分けるのか?

| メリット | 説明 |
|---|---|
| **専門化** | 各エージェントに「役割」「使えるツール」「プロンプト」を集中して設定でき、精度が上がる |
| **責務分離** | 「情報収集役」「文書化役」など、ソフトウェア工学の単一責任原則と同じ発想 |
| **再利用** | できあがった各エージェントは他のチーム構成にも組み込める |
| **デバッグ性** | 失敗したのが誰のせいか切り分けやすい |
| **プロンプトの簡潔さ** | 1 体に全部詰め込むと長くなるシステムプロンプトを分割できる |

### 2 つのアーキテクチャ

```
Supervisor 型 (本章で実装)         Swarm 型 (発展課題)
     ┌────────────┐                    ┌──────────┐
     │ Supervisor │ ←──→             ┌→│ Agent A  │←─┐
     └────┬─┬─────┘                  │ └──────────┘  │
          ↓ ↓                        │      ↕         │
      ┌───┘ └───┐                    │ ┌──────────┐  │
      ↓         ↓                    └→│ Agent B  │←─┘
    Worker A  Worker B                 └──────────┘
```

**Supervisor 型**: 中央の Supervisor がワーカーを呼び分ける。フローが追いやすく、初学者向け。
**Swarm 型**: エージェント同士が直接 hand-off。柔軟だが複雑。

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

# .env を読み込む(これまでの章と同じ)。LangSmith のトレースもここで有効になる
load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]
EMBED_MODEL_ID = os.environ["BEDROCK_EMBED_MODEL_ID"]  # 06章のChroma再ロードに使う

print("chat :", CHAT_MODEL_ID)
print("embed:", EMBED_MODEL_ID)

In [ ]:
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings

# llm:        各エージェント(researcher / writer)と Supervisor が共通で使うチャットモデル
# embeddings: 06章で作った Chroma を再ロードする際に必要(検索クエリのベクトル化に使う)
llm = ChatBedrockConverse(model=CHAT_MODEL_ID, region_name=AWS_REGION, temperature=0)
embeddings = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

## 1. Researcher エージェント

06 章で作った Chroma(`06_rag_advanced/.chroma_v2`)を **検索ツール** として使い、社内文書から情報を集める役割です。

**やること**:
1. 既存の Chroma を読み込む(再 embed は不要、永続化済みデータをそのまま使う)
2. その VectorStore を呼ぶ薄いラッパ関数を `@tool` でツール化
3. `create_react_agent` でツールを使う Researcher を作成

In [ ]:
from langchain_chroma import Chroma
from langchain_core.tools import tool

# 06章で生成済みの .chroma_v2 を再ロード(persist_directory に既存データがあれば自動で読まれる)
vectorstore = Chroma(
    persist_directory="../06_rag_advanced/.chroma_v2",
    embedding_function=embeddings,
)

# 件数を確認。0 なら 06 章の Notebook を一度通しで実行してから戻ってきてください
n = len(vectorstore.get()["ids"])
print(f"VectorStore 件数: {n}")
assert n > 0, "06章のNotebookを実行してから来てください(.chroma_v2 が空です)"

# 06章の retriever を関数化してツール化
@tool
def retrieve_company_docs(query: str) -> str:
    """社内文書(社員ハンドブック・製品FAQ・技術文書)から、クエリに関連する情報を最大5件取得して返す。"""
    docs = vectorstore.similarity_search(query, k=5)
    return "\n\n---\n\n".join(d.page_content for d in docs)

# 動作確認
print(retrieve_company_docs.invoke("テレワーク 通信費")[:200], "...")

In [ ]:
from langgraph.prebuilt import create_react_agent

# Researcher エージェント: retrieve_company_docs ツールを持つ
# name は Supervisor が "このエージェントに渡そう" と判断するための識別子。必ず設定する
researcher = create_react_agent(
    llm,
    tools=[retrieve_company_docs],
    prompt=(
        "あなたは社内文書のリサーチ専門エージェント `researcher` です。\n"
        "渡されたトピックについて、retrieve_company_docs ツールで関連情報を集めてください。\n"
        "必要に応じてクエリを変えて2〜3回検索しても構いません。\n"
        "集めた情報を整理して、要点を箇条書きで返してください。\n"
        "自分で文章をきれいに整える必要はありません(後段のWriterが担当します)。"
    ),
    name="researcher",  # ← Supervisor から指名されるための識別子
)

## 2. Writer エージェント

Researcher が集めた情報を、読みやすい文章にまとめる役割。ツールは持たず、純粋に LLM の文章生成能力だけ使います。

In [ ]:
# Writer エージェント: ツールなし
writer = create_react_agent(
    llm,
    tools=[],  # ツールなしでも create_react_agent は使える(LLM 1回呼んで終わるだけのエージェントになる)
    prompt=(
        "あなたは文章作成専門エージェント `writer` です。\n"
        "渡された情報(箇条書きや断片)をもとに、依頼された文字数・読者層に合わせた\n"
        "自然な日本語の文章を作成してください。\n"
        "資料に書かれていない情報を勝手に補わないでください。"
    ),
    name="writer",
)

## 3. Supervisor を構築

`langgraph-supervisor` の `create_supervisor` を使うと、複数の compiled エージェントを束ねた **トップレベルのグラフ** ができます。

**仕組み**:
- Supervisor は内部的に「各ワーカーを呼ぶ hand-off ツール」を自動で持つ
- ユーザー入力を受けたら、Supervisor の LLM が「次は誰に渡すか、それとも自分で答えて終わるか」を判断
- hand-off ツールを呼ぶ → 該当のワーカーエージェントに制御が移る
- ワーカーが応答を返すと、再び Supervisor に戻ってきて次を判断、を繰り返す

In [ ]:
from langgraph_supervisor import create_supervisor

# Supervisor グラフを作成
# - agents: 配下のワーカーのリスト
# - model: Supervisor 自身が使う LLM(ワーカーと同じでも別でもOK)
# - prompt: Supervisor の行動指針
supervisor_app = create_supervisor(
    agents=[researcher, writer],
    model=llm,
    prompt=(
        "あなたはチームの司令塔(Supervisor)です。\n"
        "配下のエージェント:\n"
        "  - researcher: 社内文書を検索して情報を集める専門家\n"
        "  - writer: 集めた情報を読みやすい文章にまとめる専門家\n"
        "\n"
        "ユーザーの依頼を分析し、必要なら researcher → writer の順に処理してください。\n"
        "単に挨拶や雑談など情報収集が要らない場合は、自分で直接答えても構いません。\n"
        "最終回答はユーザーに分かりやすい形でまとめて返してください。"
    ),
).compile()  # 注: create_supervisor は builder を返すので、 .compile() を忘れずに

### 構造を確認

In [ ]:
# Supervisor の内部構造を Mermaid で見る
# (supervisor ノード + researcher ノード + writer ノードがあり、相互に遷移可能なグラフになる)
print(supervisor_app.get_graph().draw_mermaid())

## 4. 実行: 単発の質問

1 回目はシンプルに `.invoke()` で。最終回答だけ確認します。

In [ ]:
question = "テレワークの規程について、新入社員向けに 200 字程度で説明して。"

result = supervisor_app.invoke({
    "messages": [{"role": "user", "content": question}]
})

# 最終応答だけ表示
print("=== 最終回答 ===")
print(result["messages"][-1].content)

## 5. 実行: ストリームで誰が今喋っているか観察

`.stream(...)` で **各ステップの状態スナップショット** を順に受け取れます。
それぞれの message を見ることで「Supervisor → Researcher → Writer」の流れが目で追えます。

In [ ]:
question = "SmartLogger Pro のスタンダード版を社内で導入するときの、料金とサポート範囲を、購買担当者向けに分かりやすくまとめて。"

# stream_mode="updates" だと、各ステップで「どのノードが何を出力したか」が dict で届く
for step in supervisor_app.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="updates",
):
    # step は {ノード名: そのノードの出力} の dict
    for node_name, node_output in step.items():
        print(f"\n===== ノード: {node_name} =====")
        if "messages" in node_output:
            # そのステップで追加されたメッセージを表示
            for m in node_output["messages"]:
                m.pretty_print()

## 6. LangSmith でトレースツリーを確認

`.env` に `LANGSMITH_API_KEY` を設定していれば、上の実行が自動でトレースされています。
ダッシュボードでは以下のようなツリーが見えるはずです:

```
supervisor (top-level)
├─ supervisor → LLM呼び出し           ← 「最初は researcher に渡そう」と判断
├─ researcher
│   ├─ LLM呼び出し                    ← 「retrieve_company_docs を使おう」
│   ├─ retrieve_company_docs (tool)   ← ベクトル検索結果
│   └─ LLM呼び出し                    ← 結果を整理
├─ supervisor → LLM呼び出し           ← 「次は writer に渡そう」と判断
└─ writer
    └─ LLM呼び出し                    ← 最終文章を生成
```

**ポイント**:
- 各エージェント呼び出しが入れ子のスパンとして見える
- どのエージェントがいくら時間を使い、いくらトークンを消費したかが分かる
- マルチエージェントは「失敗したのが誰のせいか」が問題になりがちだが、LangSmith があれば一目瞭然

## まとめ

- **マルチエージェント** は「専門化」「責務分離」「再利用」のために有効
- **Supervisor パターン**: 中央の司令塔が配下のワーカーを呼び分ける。最も初心者向け
- `langgraph-supervisor` + `create_react_agent` を組み合わせれば、最小コードで構築可能
- 各エージェントには **`name`** と専用 **`prompt`** を必ず設定する
- LangSmith でエージェント間の流れと責任範囲を可視化できる

### 発展トピック(本章スコープ外)

- **Swarm パターン**: エージェント同士が直接 hand-off。`langgraph-swarm` パッケージ
- **階層型 Supervisor**: Supervisor の下にさらに Supervisor を置く(大規模チーム構成)
- **動的なツール選択**: ワーカーの数が多い場合、Supervisor の判断にもツール選択 LLM の精度がボトルネックに
- **コスト最適化**: Supervisor は軽量モデル(Haiku 等)、ワーカーは適材適所、と使い分ける

次は [08 MCP連携](../08_mcp/) で、外部の MCP サーバが提供するツールをエージェントに組み込みます。